In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import joblib
import pandas as pd

from src.features import (
    add_head_to_head_features,
    add_team_form_features,
    add_team_scoring_features,
    add_team_strategy_features,
    add_venue_features,
)

In [3]:
historical_matches = pd.read_csv("../data/processed/match_by_match.csv")
model = joblib.load("../models/production/no_score/best_model.pkl")
features_columns = joblib.load("../data/splits/no_score/X_test.pkl").columns.tolist()

In [4]:
new_match = {
    "match_id": 1535466,
    "date": "2027-04-01",
    "venue": "MA Chidambaram Stadium",
    "team1": "Chennai Super Kings",
    "team2": "Mumbai Indians",
    "toss_winner": "Chennai Super Kings",
    "toss_decision": "bat"
}

In [5]:
combined = pd.concat(
    [historical_matches, pd.DataFrame([new_match])],
    ignore_index=True
)

In [6]:
combined.iloc[[-1]]

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,team2_score,team1_players,team2_players
1243,1535466,2027-04-01,MA Chidambaram Stadium,Chennai Super Kings,Mumbai Indians,Chennai Super Kings,bat,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
def build_match_features(match_df):
    match_df["toss_winner_slot"] = (match_df["toss_winner"] == match_df["team1"]).astype(int)

    match_df = add_head_to_head_features(match_df)
    match_df = add_team_form_features(match_df)
    match_df = add_team_scoring_features(match_df)
    match_df = add_team_strategy_features(match_df)
    match_df = add_venue_features(match_df)

    return match_df

model_ready_dataset = build_match_features(combined)

Added head_to_head_features:
 - team1_total_wins_against_team2
 - team2_total_wins_against_team1
 - team1_wins_against_team2_last_three
 - team2_wins_against_team1_last_three
Added team_form_features:
 - team1_form_last_5
 - team2_form_last_5
Added team_scoring_features:
 - team1_last_5_avg_score
 - team1_last_5_runs_conceded
 - team2_last_5_avg_score
 - team2_last_5_runs_conceded
Added team_strategy_features:
 - team1_chasing_win_rate
 - team1_defending_win_rate
 - team2_chasing_win_rate
 - team2_defending_win_rate
Added venue_features:
 - venue_avg_score
 - chasing_win_rate_venue
 - team1_win_rate_at_venue
 - team2_win_rate_at_venue


In [8]:
X = model_ready_dataset.iloc[[-1]][features_columns]
X

,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,toss_winner_slot
1243,20,21,2,1,2,2,174.8,191.2,184.4,189.6,0.606,0.59,0.529,0.536,0.0,0.0,0.0,0.0,1


In [9]:
prediction = model.predict(X)
prob = model.predict_proba(X)

In [10]:
winner = (
    new_match["team1"]
    if prediction[0] == 0
    else new_match["team2"]
)

print("========== Match Prediction ==========")
print(f"Team 1 : {new_match['team1']}")
print(f"Team 2 : {new_match['team2']}")
print(f"Venue  : {new_match['venue']}")
print(f"Toss Winner : {new_match['toss_winner']}")
print(f"Toss Decision : {new_match['toss_decision']}")
print("--------------------------------------")
print(f"Predicted Winner : {winner}")
print(f"Probability of {new_match['team1']} winning : {prob[0][0]:.2%}")
print(f"Probability of {new_match['team2']} winning : {prob[0][1]:.2%}")
print("======================================")

========== Match Prediction ==========
Team 1 : Chennai Super Kings
Team 2 : Mumbai Indians
Venue  : MA Chidambaram Stadium
Toss Winner : Chennai Super Kings
Toss Decision : bat
--------------------------------------
Predicted Winner : Mumbai Indians
Probability of Chennai Super Kings winning : 35.45%
Probability of Mumbai Indians winning : 64.55%


In [14]:
model = joblib.load("../models/production/with_score/best_model.pkl")
features_columns = joblib.load("../data/splits/with_score/X_test.pkl").columns.tolist()
features_columns

['team1_score',
 'team1_total_wins_against_team2',
 'team2_total_wins_against_team1',
 'team1_wins_against_team2_last_three',
 'team2_wins_against_team1_last_three',
 'team1_form_last_5',
 'team2_form_last_5',
 'team1_last_5_avg_score',
 'team1_last_5_runs_conceded',
 'team2_last_5_avg_score',
 'team2_last_5_runs_conceded',
 'team1_chasing_win_rate',
 'team1_defending_win_rate',
 'team2_chasing_win_rate',
 'team2_defending_win_rate',
 'venue_avg_score',
 'chasing_win_rate_venue',
 'team1_win_rate_at_venue',
 'team2_win_rate_at_venue',
 'toss_winner_slot']

In [21]:
X['team1_score'] = 150
X.insert(0, "team1_score", X.pop("team1_score"))

In [22]:
prediction = model.predict(X)
prob = model.predict_proba(X)

In [23]:
winner = (
    new_match["team1"]
    if prediction[0] == 0
    else new_match["team2"]
)

print("========== Match Prediction ==========")
print(f"Team 1 : {new_match['team1']}")
print(f"Team 2 : {new_match['team2']}")
print(f"Venue  : {new_match['venue']}")
print(f"Toss Winner : {new_match['toss_winner']}")
print(f"Toss Decision : {new_match['toss_decision']}")
print("--------------------------------------")
print(f"Predicted Winner : {winner}")
print(f"Probability of {new_match['team1']} winning : {prob[0][0]:.2%}")
print(f"Probability of {new_match['team2']} winning : {prob[0][1]:.2%}")
print("======================================")

========== Match Prediction ==========
Team 1 : Chennai Super Kings
Team 2 : Mumbai Indians
Venue  : MA Chidambaram Stadium
Toss Winner : Chennai Super Kings
Toss Decision : bat
--------------------------------------
Predicted Winner : Chennai Super Kings
Probability of Chennai Super Kings winning : 62.99%
Probability of Mumbai Indians winning : 37.01%
